# Instance Segmentation Fine-Tuning

Direct Graha/Lunar-FM true instance segmentation using TerraTorch `ObjectDetectionTask` with `framework="mask-rcnn"`.

In [ ]:
# Notebook imports
# Generated from standard/third-party imports used throughout this notebook.
import sys
from argparse import Namespace
from lightning.pytorch import seed_everything
from pathlib import Path


## Config

In [ ]:

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "instance_seg_finetuning.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

SIMLINK_DEST = None
DATA_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "instance_seg_finetuning"
PRETRAIN_DIR = None
GRAHA_INPUT_MODALITY_MODE = "new-wac"  # Use "vis-uv" to reuse pretrained 5-band vis + 2-band uv modalities.
GRAHA_VIS_UV_MERGE_METHOD = "mean"
LIGHTNING_CHECKPOINT = None

CROP_SIZE = 256
STATS_BATCH_SIZE = 16
BATCH_SIZE = 2
NUM_WORKERS = 10
MAX_EPOCHS = 1

BACKBONE_LR = 5.0e-5
HEAD_LR = 2.0e-4
LAYER_DECAY = 0.75
WEIGHT_DECAY = 0.05
WARMUP_STEPS = 500
ANCHOR_SIZES = [[8], [16], [32], [64]]
ANCHOR_ASPECT_RATIOS = [0.5, 1.0, 2.0]
SCORE_THRESHOLD = 0.5
PLOT_PREDICTIONS = True
PREDICTION_SPLIT = "val"
PREDICTION_N_SAMPLES = 5
PREDICTION_SCORE_THRESHOLD = 0.5
MASK_SHIFT = (0, 0)  # (x_pixels, y_pixels): positive moves labels right/down
SEED = 42

RUN_FIT = False
LOSS_SMOKE_ONLY = False

## Environment

In [ ]:

SCRIPTS_PY_DIR = LFM_ROOT / "scripts" / "python"
for import_path in [LFM_ROOT, SCRIPTS_PY_DIR]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))


from lfm.full_model.all_tasks.utils import create_timestamped_output_dir
from lfm.full_model.all_tasks.utils.utils import ensure_data_symlink

from lfm.full_model.inst_seg import instance_seg_finetuning as workflow

workflow.configure_proj_environment()
ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

## Build Config

In [ ]:
args = Namespace(
    simlink_dest=SIMLINK_DEST,
    data_root=str(DATA_ROOT) if DATA_ROOT is not None else None,
    base_output_dir=str(BASE_OUTPUT_DIR) if BASE_OUTPUT_DIR is not None else None,
    pretrain_dir=str(PRETRAIN_DIR) if PRETRAIN_DIR is not None else None,
    graha_input_modality_mode=GRAHA_INPUT_MODALITY_MODE,
    graha_vis_uv_merge_method=GRAHA_VIS_UV_MERGE_METHOD,
    lightning_checkpoint=str(LIGHTNING_CHECKPOINT) if LIGHTNING_CHECKPOINT is not None else None,
    crop_size=CROP_SIZE,
    stats_batch_size=STATS_BATCH_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_epochs=MAX_EPOCHS,
    backbone_lr=BACKBONE_LR,
    head_lr=HEAD_LR,
    layer_decay=LAYER_DECAY,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    anchor_sizes=ANCHOR_SIZES,
    anchor_aspect_ratios=ANCHOR_ASPECT_RATIOS,
    score_threshold=SCORE_THRESHOLD,
    plot_predictions=PLOT_PREDICTIONS,
    prediction_split=PREDICTION_SPLIT,
    prediction_n_samples=PREDICTION_N_SAMPLES,
    prediction_score_threshold=PREDICTION_SCORE_THRESHOLD,
    mask_shift=MASK_SHIFT,
    seed=SEED,
    no_fit=not RUN_FIT,
    loss_smoke_only=LOSS_SMOKE_ONLY,
)

config = workflow.build_config(args)
workflow.configure_python_paths(config)
workflow.print_config(config)
workflow.validate_required_paths(config)
deps = workflow.import_project_dependencies()

## Output Directory

In [ ]:
output_dir = create_timestamped_output_dir(config.base_output_dir)
(output_dir / "checkpoints" / "full_model").mkdir(parents=True, exist_ok=True)
workflow.save_config(config, output_dir)
seed_everything(config.seed)
print("Output directory:", output_dir)

## Training Stats

In [ ]:
datamodule_cls = deps["LunarObjectDetectionInstanceMaskDatamodule"]
means, stds = workflow.calculate_train_stats(config, datamodule_cls)

## Datamodule

In [ ]:
datamodule = workflow.create_datamodule(config, datamodule_cls, means, stds)
sample_batch = workflow.inspect_batch(datamodule)

## Graha Mask R-CNN Task

In [ ]:
task_cls = workflow.make_downstream_object_detection_task_class(deps["LunarObjectDetectionTask"])
task = workflow.create_task(config, task_cls, sample_batch)
print(type(task.model))

## Loss Smoke Test

In [ ]:
workflow.run_loss_smoke(task, sample_batch)

## Train

In [ ]:
if RUN_FIT:
    trainer = workflow.create_trainer(config, output_dir)
    ckpt_path = str(config.lightning_checkpoint) if config.lightning_checkpoint is not None else None
    trainer.fit(task, datamodule=datamodule, ckpt_path=ckpt_path)
else:
    print("RUN_FIT is False; skipping trainer.fit().")

## Prediction Plots

In [ ]:
if PLOT_PREDICTIONS:
    if not RUN_FIT and config.lightning_checkpoint is not None:
        workflow.load_lightning_checkpoint_state(task, config.lightning_checkpoint)
    workflow.save_instance_prediction_plots(task, datamodule, config, output_dir)
else:
    print("PLOT_PREDICTIONS is False; skipping prediction plots.")